# Imports

In [1]:
import os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Loader

In [2]:
CLASSES = ["packaging", "drinking", "idle"]

root_dir = "../data/"
samples = []
for dir_name in os.listdir(root_dir):
    label = dir_name.split('_')[0]
    full_path = os.path.join(root_dir, dir_name)
    if os.path.isdir(full_path):
        samples.append((full_path, label))

imu_data = pd.DataFrame(columns=['gyro_x','gyro_y','gyro_z','accel_x','accel_y','accel_z','label'])
for s in samples:
    path, label_str = s
    imu = pd.read_csv(os.path.join(path, 'out.csv'))
    imu = imu.iloc[:, :6]  
    imu['label'] = CLASSES.index(label_str)
    # Select only the IMU data columns (assuming first 6 are IMU related)
    imu_data = imu_data._append(imu, ignore_index=True)
imu_data['label'].unique()

/var/folders/0y/4sn48ws54fvc3hldm5bsrgj00000gn/T/ipykernel_63047/2707067197.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  imu_data = imu_data._append(imu, ignore_index=True)


array([2, 1, 0], dtype=object)

# Split


In [3]:
imu_data['label'] = imu_data['label'].astype('category')
X = imu_data.iloc[:, :-1]  
y = imu_data.iloc[:, -1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train


In [4]:
from sklearn.metrics import accuracy_score, classification_report
rf_classifier = RandomForestClassifier(n_estimators=100,max_depth=10, min_samples_split=5)
rf_classifier.fit(X_train, y_train)
y_pred = rf_classifier.predict(X_test)
print(classification_report(y_test,y_pred))

accuracy = rf_classifier.score(X_test, y_test)
print("Accuracy:", accuracy)

              precision    recall  f1-score   support

           0       0.91      0.94      0.93       234
           1       0.92      0.89      0.91       171
           2       1.00      0.99      1.00       321

    accuracy                           0.95       726
   macro avg       0.94      0.94      0.94       726
weighted avg       0.95      0.95      0.95       726

Accuracy: 0.9517906336088154


# Eval

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
confusion_mat = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:")
print(confusion_mat)

param_grid = {
    'min_samples_split': [2, 5, 7],
    'max_depth': [3, 5, 7],
    'n_estimators': [100,200,500]
}
number_of_cross_validation_folds=5

grid_search = GridSearchCV(estimator=rf_classifier, param_grid=param_grid, cv=5)
grid_search.fit(X, y)

print("Best Parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)

Accuracy: 0.9517906336088154
Precision: 0.9521689106954638
Recall: 0.9517906336088154
F1 Score: 0.9518753489957231
Confusion Matrix:
[[220  14   0]
 [ 18 153   0]
 [  3   0 318]]
Best Parameters: {'max_depth': 7, 'min_samples_split': 5, 'n_estimators': 100}
Best Score: 0.9368496247743895


In [ ]:
import torch

input_df = pd.read_csv('/Users/sanjivjha/Developer/ubicomp-capstone/collected/1708693371out.csv')

import torch.nn.functional as F
CLASSES = ["packaging", "drinking", "idle"]
output_data = pd.DataFrame([], columns=['Timestamp', 'Temperature', 'Activity'])
imu_dataset = input_df.iloc[:,:6]
for i in imu_dataset.index.values.tolist():
    output = rf_classifier.predict_proba([imu_dataset.iloc[i]])
    predictions = output.argmax(axis=1)
    predicted_classes = [rf_classifier.classes_[pred] for pred in predictions][0]
    output_data.loc[len(output_data.index)] = [pd.to_datetime(input_df.loc[i]['timestamp'], unit='s'), round(input_df.loc[i]['temp'],2) , CLASSES[predicted_classes]]
    print('Timestamp = {}, Temperature = {} degree C, : Activity = {}'.format(pd.to_datetime(input_df.loc[i]['timestamp'], unit='s'), round(input_df.loc[i]['temp'],2) , CLASSES[predicted_classes]))
output_data.to_csv('../collected/rf_final_output.csv')

# Save

# Load